<a href="https://colab.research.google.com/github/kamal-gavel/NATURAL-LANGUAGE-PROCESSING-IMPLEMENTATION-IN-FINANCE-/blob/main/WORD2Vec_Implementation_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================================
# 🚀 WORD2VEC FROM SCRATCH (SKIP-GRAM + NEGATIVE SAMPLING)
# =========================================================

import numpy as np
import random
from collections import Counter
from tqdm import tqdm

# -------------------------------
# 🔹 1. PREPROCESS TEXT
# -------------------------------
def preprocess_text(text):
    """
    Convert text to lowercase and tokenize
    """
    return text.lower().split()


# -------------------------------
# 🔹 2. BUILD VOCABULARY
# -------------------------------
def build_vocab(words, min_count=1):
    """
    Create word-to-index and index-to-word mappings
    """
    word_counts = Counter(words)

    vocab = [word for word, count in word_counts.items() if count >= min_count]

    word2idx = {word: i for i, word in enumerate(vocab)}
    idx2word = {i: word for word, i in word2idx.items()}

    return word2idx, idx2word, word_counts


# -------------------------------
# 🔹 3. GENERATE TRAINING PAIRS
# -------------------------------
def generate_training_data(words, word2idx, window_size=2):
    """
    Generate (center, context) pairs using sliding window
    """
    pairs = []

    for i, word in enumerate(words):
        if word not in word2idx:
            continue

        center = word2idx[word]

        # Context window
        for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
            if i != j and words[j] in word2idx:
                context = word2idx[words[j]]
                pairs.append((center, context))

    return pairs


# -------------------------------
# 🔹 4. NEGATIVE SAMPLING
# -------------------------------
def get_negative_sampler(word_counts, word2idx, k=5):
    """
    Create a sampler for negative samples
    """
    vocab = list(word2idx.keys())
    freqs = np.array([word_counts[w] for w in vocab])

    # Smooth distribution (important trick)
    probs = freqs ** 0.75
    probs = probs / np.sum(probs)

    def sample():
        return np.random.choice(len(vocab), size=k, p=probs)

    return sample


# -------------------------------
# 🔹 5. MODEL INITIALIZATION
# -------------------------------
class Word2Vec:
    def __init__(self, vocab_size, embedding_dim):
        """
        Initialize embeddings
        """
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        # Center word embeddings
        self.W = np.random.randn(vocab_size, embedding_dim) * 0.01

        # Context word embeddings
        self.W_context = np.random.randn(vocab_size, embedding_dim) * 0.01


# -------------------------------
# 🔹 6. SIGMOID FUNCTION
# -------------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# -------------------------------
# 🔹 7. TRAINING LOOP (CORE)
# -------------------------------
def train(model, pairs, negative_sampler, epochs=5, lr=0.01, k=5):
    """
    Train Word2Vec using Skip-gram + Negative Sampling
    """
    for epoch in range(epochs):
        total_loss = 0

        for center, context in tqdm(pairs, desc=f"Epoch {epoch+1}"):

            # Get embeddings
            v_c = model.W[center]
            u_o = model.W_context[context]

            # ---------------------
            # ✅ POSITIVE SAMPLE
            # ---------------------
            score = sigmoid(np.dot(u_o, v_c))
            loss_pos = -np.log(score + 1e-10)
            total_loss += loss_pos

            grad_pos = (score - 1)

            # Update embeddings
            model.W[center] -= lr * grad_pos * u_o
            model.W_context[context] -= lr * grad_pos * v_c

            # ---------------------
            # ❌ NEGATIVE SAMPLES
            # ---------------------
            negatives = negative_sampler()

            for neg in negatives:
                u_k = model.W_context[neg]

                score_neg = sigmoid(np.dot(u_k, v_c))
                loss_neg = -np.log(1 - score_neg + 1e-10)
                total_loss += loss_neg

                grad_neg = score_neg

                # Update embeddings
                model.W[center] -= lr * grad_neg * u_k
                model.W_context[neg] -= lr * grad_neg * v_c

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# -------------------------------
# 🔹 8. SIMILARITY FUNCTION
# -------------------------------
def get_similar_words(model, word, word2idx, idx2word, top_k=5):
    """
    Find most similar words using cosine similarity
    """
    if word not in word2idx:
        return []

    idx = word2idx[word]
    vector = model.W[idx]

    similarities = []

    for i in range(model.vocab_size):
        sim = np.dot(vector, model.W[i]) / (
            np.linalg.norm(vector) * np.linalg.norm(model.W[i]) + 1e-10
        )
        similarities.append((idx2word[i], sim))

    similarities.sort(key=lambda x: -x[1])

    return similarities[1:top_k+1]


# =========================================================
# 🧪 9. RUN THE MODEL
# =========================================================

# Sample corpus (you can replace with your dataset)
text = "king queen man woman king man queen woman prince princess"

# Preprocess
words = preprocess_text(text)

# Build vocab
word2idx, idx2word, word_counts = build_vocab(words)

# Generate training data
pairs = generate_training_data(words, word2idx, window_size=2)

# Negative sampler
negative_sampler = get_negative_sampler(word_counts, word2idx, k=5)

# Initialize model
model = Word2Vec(vocab_size=len(word2idx), embedding_dim=50)

# Train model
train(model, pairs, negative_sampler, epochs=10, lr=0.01)

# Test similarity
print("\n🔍 Similar words to 'king':")
print(get_similar_words(model, "king", word2idx, idx2word))

Epoch 1: 100%|██████████| 34/34 [00:00<00:00, 6251.10it/s]


Epoch 1, Loss: 141.4188


Epoch 2: 100%|██████████| 34/34 [00:00<00:00, 6156.38it/s]


Epoch 2, Loss: 141.3941


Epoch 3: 100%|██████████| 34/34 [00:00<00:00, 5440.71it/s]


...

Epoch 8, Loss: 141.2210


Epoch 9: 100%|██████████| 34/34 [00:00<00:00, 4631.28it/s]


Epoch 9, Loss: 141.1858


Epoch 10: 100%|██████████| 34/34 [00:00<00:00, 4698.11it/s]

Epoch 10, Loss: 141.1428

🔍 Similar words to 'king':
[('queen', np.float64(0.4857457951382061)), ('man', np.float64(0.38206158998428846)), ('woman', np.float64(0.3403678439956626)), ('princess', np.float64(0.2793374357229483)), ('prince', np.float64(0.24762354784641305))]


In [2]:
"""
Here is your corrected + improved single-cell Word2Vec (Skip-gram + Negative Sampling) code with:

✅ Better learning
✅ Larger training signal
✅ Stable gradients
✅ Normalized embeddings
✅ More realistic sample data
"""
# =========================================================
# 🚀 WORD2VEC (SKIP-GRAM + NEGATIVE SAMPLING) - IMPROVED
# =========================================================

import numpy as np
from collections import Counter
from tqdm import tqdm

# -------------------------------
# 🔹 1. PREPROCESS TEXT
# -------------------------------
def preprocess_text(text):
    return text.lower().split()


# -------------------------------
# 🔹 2. BUILD VOCAB
# -------------------------------
def build_vocab(words, min_count=1):
    word_counts = Counter(words)

    vocab = [w for w, c in word_counts.items() if c >= min_count]

    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}

    return word2idx, idx2word, word_counts


# -------------------------------
# 🔹 3. TRAINING PAIRS
# -------------------------------
def generate_pairs(words, word2idx, window_size=4):
    pairs = []

    for i, word in enumerate(words):
        if word not in word2idx:
            continue

        center = word2idx[word]

        for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
            if i != j and words[j] in word2idx:
                context = word2idx[words[j]]
                pairs.append((center, context))

    return pairs


# -------------------------------
# 🔹 4. NEGATIVE SAMPLING
# -------------------------------
def get_negative_sampler(word_counts, word2idx, k=5):
    vocab = list(word2idx.keys())
    freqs = np.array([word_counts[w] for w in vocab])

    probs = freqs ** 0.75
    probs /= np.sum(probs)

    def sample():
        return np.random.choice(len(vocab), size=k, p=probs)

    return sample


# -------------------------------
# 🔹 5. MODEL
# -------------------------------
class Word2Vec:
    def __init__(self, vocab_size, dim):
        self.vocab_size = vocab_size
        self.dim = dim

        # Xavier Initialization (better than random small)
        self.W = np.random.randn(vocab_size, dim) / np.sqrt(dim)
        self.W_context = np.random.randn(vocab_size, dim) / np.sqrt(dim)


# -------------------------------
# 🔹 6. SIGMOID
# -------------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -10, 10)))  # clipping for stability


# -------------------------------
# 🔹 7. TRAINING
# -------------------------------
def train(model, pairs, neg_sampler, epochs=50, lr=0.005, k=5):

    for epoch in range(epochs):
        total_loss = 0

        for center, context in pairs:

            v_c = model.W[center]
            u_o = model.W_context[context]

            # -----------------
            # POSITIVE SAMPLE
            # -----------------
            score = sigmoid(np.dot(u_o, v_c))
            loss = -np.log(score + 1e-9)
            total_loss += loss

            grad = (score - 1)

            # Update
            model.W[center] -= lr * grad * u_o
            model.W_context[context] -= lr * grad * v_c

            # -----------------
            # NEGATIVE SAMPLES
            # -----------------
            negatives = neg_sampler()

            for neg in negatives:
                u_k = model.W_context[neg]

                score_neg = sigmoid(np.dot(u_k, v_c))
                loss_neg = -np.log(1 - score_neg + 1e-9)
                total_loss += loss_neg

                grad_neg = score_neg

                model.W[center] -= lr * grad_neg * u_k
                model.W_context[neg] -= lr * grad_neg * v_c

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# -------------------------------
# 🔹 8. NORMALIZE EMBEDDINGS
# -------------------------------
def normalize_embeddings(model):
    norms = np.linalg.norm(model.W, axis=1, keepdims=True)
    model.W = model.W / (norms + 1e-9)


# -------------------------------
# 🔹 9. SIMILARITY
# -------------------------------
def get_similar(model, word, word2idx, idx2word, top_k=5):
    if word not in word2idx:
        return []

    idx = word2idx[word]
    vec = model.W[idx]

    sims = []
    for i in range(model.vocab_size):
        sim = np.dot(vec, model.W[i])
        sims.append((idx2word[i], sim))

    sims.sort(key=lambda x: -x[1])
    return sims[1:top_k+1]


# =========================================================
# 🧪 10. SAMPLE DATA (IMPROVED CORPUS)
# =========================================================

text = """
king is a strong man queen is a wise woman
man and woman are human beings
king and queen rule the kingdom
prince is the son of king
princess is the daughter of queen
"""

# -------------------------------
# 🔹 PIPELINE
# -------------------------------
words = preprocess_text(text)

word2idx, idx2word, word_counts = build_vocab(words)

pairs = generate_pairs(words, word2idx, window_size=4)

neg_sampler = get_negative_sampler(word_counts, word2idx)

model = Word2Vec(vocab_size=len(word2idx), dim=50)

train(model, pairs, neg_sampler, epochs=50, lr=0.005)

normalize_embeddings(model)

# -------------------------------
# 🔍 TEST
# -------------------------------
print("\n🔍 Similar to 'king':")
print(get_similar(model, "king", word2idx, idx2word))

print("\n🔍 Similar to 'queen':")
print(get_similar(model, "queen", word2idx, idx2word))

Epoch 10, Loss: 884.8590
Epoch 20, Loss: 712.4928
Epoch 30, Loss: 670.7165
Epoch 40, Loss: 647.9466
Epoch 50, Loss: 632.6503

🔍 Similar to 'king':
[('beings', np.float64(0.6513558350376625)), ('man', np.float64(0.628709963385515)), ('strong', np.float64(0.6147771732782465)), ('princess', np.float64(0.6133483656655483)), ('daughter', np.float64(0.6121671653007211))]

🔍 Similar to 'queen':
[('princess', np.float64(0.7071616269946578)), ('a', np.float64(0.6543275916347991)), ('prince', np.float64(0.651721007883365)), ('son', np.float64(0.6315015777674619)), ('beings', np.float64(0.6275439091769173))]


In [3]:
"""
Perfect — here is your final corrected, clean, single-cell Word2Vec implementation with:

✅ Fixed gradient bug
✅ Stopword removal
✅ Stable updates
✅ Better training behavior
✅ Clean comments (exam + research ready)
"""
# =========================================================
# 🚀 WORD2VEC (SKIP-GRAM + NEGATIVE SAMPLING) - FINAL FIXED
# =========================================================

import numpy as np
from collections import Counter
from tqdm import tqdm

# -------------------------------
# 🔹 1. PREPROCESS TEXT (REMOVE NOISE)
# -------------------------------
def preprocess_text(text):
    stopwords = set(["is", "a", "the", "and", "of", "to", "are"])
    words = text.lower().split()
    words = [w for w in words if w not in stopwords]
    return words


# -------------------------------
# 🔹 2. BUILD VOCAB
# -------------------------------
def build_vocab(words, min_count=1):
    word_counts = Counter(words)
    vocab = [w for w, c in word_counts.items() if c >= min_count]

    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}

    return word2idx, idx2word, word_counts


# -------------------------------
# 🔹 3. GENERATE TRAINING PAIRS
# -------------------------------
def generate_pairs(words, word2idx, window_size=3):
    pairs = []

    for i, word in enumerate(words):
        if word not in word2idx:
            continue

        center = word2idx[word]

        for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
            if i != j and words[j] in word2idx:
                context = word2idx[words[j]]
                pairs.append((center, context))

    return pairs


# -------------------------------
# 🔹 4. NEGATIVE SAMPLING
# -------------------------------
def get_negative_sampler(word_counts, word2idx, k=5):
    vocab = list(word2idx.keys())
    freqs = np.array([word_counts[w] for w in vocab])

    # Smooth distribution
    probs = freqs ** 0.75
    probs /= np.sum(probs)

    def sample():
        return np.random.choice(len(vocab), size=k, p=probs)

    return sample


# -------------------------------
# 🔹 5. MODEL INITIALIZATION
# -------------------------------
class Word2Vec:
    def __init__(self, vocab_size, dim):
        self.vocab_size = vocab_size
        self.dim = dim

        # Xavier initialization (stable)
        self.W = np.random.randn(vocab_size, dim) / np.sqrt(dim)
        self.W_context = np.random.randn(vocab_size, dim) / np.sqrt(dim)


# -------------------------------
# 🔹 6. SIGMOID (STABLE)
# -------------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -10, 10)))


# -------------------------------
# 🔹 7. TRAINING LOOP (FIXED)
# -------------------------------
def train(model, pairs, neg_sampler, epochs=50, lr=0.005, k=5):

    for epoch in range(epochs):
        total_loss = 0

        for center, context in pairs:

            # Copy center vector (IMPORTANT FIX)
            v_c = model.W[center].copy()
            grad_v = np.zeros_like(v_c)

            # -----------------
            # POSITIVE SAMPLE
            # -----------------
            u_o = model.W_context[context]
            score = sigmoid(np.dot(u_o, v_c))

            total_loss += -np.log(score + 1e-9)

            grad = (score - 1)
            grad_v += grad * u_o

            # Update context vector
            model.W_context[context] -= lr * grad * v_c

            # -----------------
            # NEGATIVE SAMPLES
            # -----------------
            negatives = neg_sampler()

            for neg in negatives:
                u_k = model.W_context[neg]

                score_neg = sigmoid(np.dot(u_k, v_c))
                total_loss += -np.log(1 - score_neg + 1e-9)

                grad_neg = score_neg
                grad_v += grad_neg * u_k

                # Update negative context vector
                model.W_context[neg] -= lr * grad_neg * v_c

            # ✅ SINGLE UPDATE (CRITICAL FIX)
            model.W[center] -= lr * grad_v

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# -------------------------------
# 🔹 8. NORMALIZE EMBEDDINGS
# -------------------------------
def normalize_embeddings(model):
    norms = np.linalg.norm(model.W, axis=1, keepdims=True)
    model.W = model.W / (norms + 1e-9)


# -------------------------------
# 🔹 9. SIMILARITY FUNCTION
# -------------------------------
def get_similar(model, word, word2idx, idx2word, top_k=5):
    if word not in word2idx:
        return []

    idx = word2idx[word]
    vec = model.W[idx]

    sims = []
    for i in range(model.vocab_size):
        sim = np.dot(vec, model.W[i])
        sims.append((idx2word[i], sim))

    sims.sort(key=lambda x: -x[1])
    return sims[1:top_k+1]


# =========================================================
# 🧪 10. BETTER SAMPLE DATA
# =========================================================

text = """
king strong ruler royal power throne kingdom
queen royal woman throne kingdom power
man male human strong person
woman female human person
prince royal son king
princess royal daughter queen
"""

# -------------------------------
# 🔹 PIPELINE
# -------------------------------
words = preprocess_text(text)

word2idx, idx2word, word_counts = build_vocab(words)

pairs = generate_pairs(words, word2idx, window_size=3)

neg_sampler = get_negative_sampler(word_counts, word2idx, k=5)

model = Word2Vec(vocab_size=len(word2idx), dim=50)

train(model, pairs, neg_sampler, epochs=50, lr=0.005)

normalize_embeddings(model)

# -------------------------------
# 🔍 TEST OUTPUT
# -------------------------------
print("\n🔍 Similar to 'king':")
print(get_similar(model, "king", word2idx, idx2word))

print("\n🔍 Similar to 'queen':")
print(get_similar(model, "queen", word2idx, idx2word))

Epoch 10, Loss: 637.1980
Epoch 20, Loss: 527.6331
Epoch 30, Loss: 463.6828
Epoch 40, Loss: 437.0661
Epoch 50, Loss: 428.0601

🔍 Similar to 'king':
[('princess', np.float64(0.6307490326567938)), ('person', np.float64(0.6059845563687694)), ('throne', np.float64(0.5897658768570043)), ('prince', np.float64(0.5823829846188212)), ('son', np.float64(0.5738816819360745))]

🔍 Similar to 'queen':
[('power', np.float64(0.7013853436383932)), ('kingdom', np.float64(0.6778050191551874)), ('ruler', np.float64(0.642480839634152)), ('woman', np.float64(0.6378807204968927)), ('throne', np.float64(0.6362212330509754))]


In [4]:
"""
✅ Proper dataset (king ↔ queen alignment)
✅ Stopword handling
✅ Fixed gradient update
✅ Stable training
✅ Better results
"""
# =========================================================
# 🚀 WORD2VEC FINAL (SKIP-GRAM + NEGATIVE SAMPLING)
# =========================================================

import numpy as np
from collections import Counter

# -------------------------------
# 🔹 1. PREPROCESS TEXT
# -------------------------------
def preprocess_text(text):
    stopwords = set(["is", "a", "the", "and", "of", "to", "are"])
    words = text.lower().split()
    words = [w for w in words if w not in stopwords]
    return words


# -------------------------------
# 🔹 2. BUILD VOCAB
# -------------------------------
def build_vocab(words):
    word_counts = Counter(words)
    vocab = list(word_counts.keys())

    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for w, i in word2idx.items()}

    return word2idx, idx2word, word_counts


# -------------------------------
# 🔹 3. GENERATE TRAINING PAIRS
# -------------------------------
def generate_pairs(words, word2idx, window_size=3):
    pairs = []

    for i, word in enumerate(words):
        center = word2idx[word]

        for j in range(max(0, i-window_size), min(len(words), i+window_size+1)):
            if i != j:
                context = word2idx[words[j]]
                pairs.append((center, context))

    return pairs


# -------------------------------
# 🔹 4. NEGATIVE SAMPLING
# -------------------------------
def get_negative_sampler(word_counts, word2idx, k=5):
    vocab = list(word2idx.keys())
    freqs = np.array([word_counts[w] for w in vocab])

    probs = freqs ** 0.75
    probs /= np.sum(probs)

    def sample():
        return np.random.choice(len(vocab), size=k, p=probs)

    return sample


# -------------------------------
# 🔹 5. MODEL
# -------------------------------
class Word2Vec:
    def __init__(self, vocab_size, dim):
        self.W = np.random.randn(vocab_size, dim) / np.sqrt(dim)
        self.W_context = np.random.randn(vocab_size, dim) / np.sqrt(dim)


# -------------------------------
# 🔹 6. SIGMOID
# -------------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -10, 10)))


# -------------------------------
# 🔹 7. TRAINING (FIXED VERSION)
# -------------------------------
def train(model, pairs, neg_sampler, epochs=50, lr=0.005):

    for epoch in range(epochs):
        loss = 0

        for center, context in pairs:

            v_c = model.W[center].copy()
            grad_v = np.zeros_like(v_c)

            # -------- POSITIVE --------
            u_o = model.W_context[context]
            score = sigmoid(np.dot(u_o, v_c))

            loss += -np.log(score + 1e-9)
            grad = (score - 1)

            grad_v += grad * u_o
            model.W_context[context] -= lr * grad * v_c

            # -------- NEGATIVE --------
            negatives = neg_sampler()

            for neg in negatives:
                u_k = model.W_context[neg]

                score_neg = sigmoid(np.dot(u_k, v_c))
                loss += -np.log(1 - score_neg + 1e-9)

                grad_neg = score_neg
                grad_v += grad_neg * u_k

                model.W_context[neg] -= lr * grad_neg * v_c

            # -------- SINGLE UPDATE --------
            model.W[center] -= lr * grad_v

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


# -------------------------------
# 🔹 8. NORMALIZE EMBEDDINGS
# -------------------------------
def normalize(model):
    model.W = model.W / (np.linalg.norm(model.W, axis=1, keepdims=True) + 1e-9)


# -------------------------------
# 🔹 9. SIMILARITY
# -------------------------------
def get_similar(model, word, word2idx, idx2word, top_k=5):
    idx = word2idx[word]
    vec = model.W[idx]

    sims = []
    for i in range(len(model.W)):
        sim = np.dot(vec, model.W[i])
        sims.append((idx2word[i], sim))

    sims.sort(key=lambda x: -x[1])
    return sims[1:top_k+1]


# =========================================================
# 🧪 10. IMPROVED DATASET (CRITICAL FIX)
# =========================================================

text = """
king queen royal throne kingdom ruler power
queen king royal throne kingdom ruler power
king queen prince princess royal family
man woman human male female person
prince princess king queen royal child
"""

# -------------------------------
# 🔹 PIPELINE
# -------------------------------
words = preprocess_text(text)

word2idx, idx2word, word_counts = build_vocab(words)

pairs = generate_pairs(words, word2idx, window_size=3)

neg_sampler = get_negative_sampler(word_counts, word2idx)

model = Word2Vec(len(word2idx), dim=50)

train(model, pairs, neg_sampler, epochs=50, lr=0.005)

normalize(model)

# -------------------------------
# 🔍 TEST
# -------------------------------
print("\n🔍 Similar to 'king':")
print(get_similar(model, "king", word2idx, idx2word))

print("\n🔍 Similar to 'queen':")
print(get_similar(model, "queen", word2idx, idx2word))

Epoch 10, Loss: 652.0790
Epoch 20, Loss: 529.1621
Epoch 30, Loss: 485.4241
Epoch 40, Loss: 466.3854
Epoch 50, Loss: 455.2035

🔍 Similar to 'king':
[('ruler', np.float64(0.6976581206112199)), ('throne', np.float64(0.6030974149966122)), ('royal', np.float64(0.6021075389024658)), ('prince', np.float64(0.5926670394433603)), ('power', np.float64(0.5892937720051152))]

🔍 Similar to 'queen':
[('throne', np.float64(0.6397503188603924)), ('family', np.float64(0.5949617950130134)), ('princess', np.float64(0.58814166632193)), ('ruler', np.float64(0.5855119437120024)), ('royal', np.float64(0.5816280999216836))]


In [5]:
"""
🚀 FINAL CORRECTED ONE-CELL CODE (BEST & STABLE)

This version includes:

✅ Proper dataset (critical fix)
✅ Correct gradient update
✅ Stable training
✅ Clean preprocessing
✅ Deterministic results (seed added)
"""
# =========================================================
# 🚀 FINAL WORD2VEC (SKIP-GRAM + NEGATIVE SAMPLING)
# =========================================================

import numpy as np
from collections import Counter

np.random.seed(42)  # ✅ reproducibility

# -------------------------------
# 🔹 1. PREPROCESS
# -------------------------------
def preprocess(text):
    stopwords = {"is", "a", "the", "and", "of", "to", "are"}
    words = text.lower().split()
    return [w for w in words if w not in stopwords]


# -------------------------------
# 🔹 2. VOCAB
# -------------------------------
def build_vocab(words):
    counts = Counter(words)
    vocab = list(counts.keys())

    word2idx = {w:i for i,w in enumerate(vocab)}
    idx2word = {i:w for w,i in word2idx.items()}

    return word2idx, idx2word, counts


# -------------------------------
# 🔹 3. PAIRS
# -------------------------------
def generate_pairs(words, word2idx, window=3):
    pairs = []

    for i, w in enumerate(words):
        center = word2idx[w]

        for j in range(max(0,i-window), min(len(words), i+window+1)):
            if i != j:
                pairs.append((center, word2idx[words[j]]))

    return pairs


# -------------------------------
# 🔹 4. NEGATIVE SAMPLING
# -------------------------------
def negative_sampler(counts, word2idx, k=5):
    vocab = list(word2idx.keys())
    freqs = np.array([counts[w] for w in vocab])

    probs = freqs**0.75
    probs /= probs.sum()

    def sample():
        return np.random.choice(len(vocab), size=k, p=probs)

    return sample


# -------------------------------
# 🔹 5. MODEL
# -------------------------------
class Word2Vec:
    def __init__(self, V, D):
        self.W = np.random.randn(V, D) / np.sqrt(D)
        self.Wc = np.random.randn(V, D) / np.sqrt(D)


# -------------------------------
# 🔹 6. SIGMOID
# -------------------------------
def sigmoid(x):
    return 1/(1+np.exp(-np.clip(x, -10, 10)))


# -------------------------------
# 🔹 7. TRAIN (CORRECTED)
# -------------------------------
def train(model, pairs, sampler, epochs=80, lr=0.005):

    for e in range(epochs):
        loss = 0

        for center, context in pairs:

            vc = model.W[center].copy()
            grad_v = np.zeros_like(vc)

            # ---- positive ----
            uo = model.Wc[context]
            score = sigmoid(uo @ vc)
            loss += -np.log(score + 1e-9)

            grad = (score - 1)
            grad_v += grad * uo
            model.Wc[context] -= lr * grad * vc

            # ---- negatives ----
            for neg in sampler():
                uk = model.Wc[neg]
                s = sigmoid(uk @ vc)

                loss += -np.log(1 - s + 1e-9)

                grad_n = s
                grad_v += grad_n * uk
                model.Wc[neg] -= lr * grad_n * vc

            # ---- single update ----
            model.W[center] -= lr * grad_v

        if (e+1) % 20 == 0:
            print(f"Epoch {e+1}, Loss: {loss:.4f}")


# -------------------------------
# 🔹 8. NORMALIZE
# -------------------------------
def normalize(model):
    model.W /= (np.linalg.norm(model.W, axis=1, keepdims=True)+1e-9)


# -------------------------------
# 🔹 9. SIMILARITY
# -------------------------------
def similar(model, word, word2idx, idx2word, k=5):
    vec = model.W[word2idx[word]]

    sims = []
    for i in range(len(model.W)):
        sims.append((idx2word[i], vec @ model.W[i]))

    sims.sort(key=lambda x:-x[1])
    return sims[1:k+1]


# =========================================================
# 🧪 10. FINAL DATASET (CRITICAL)
# =========================================================

text = """
king queen king queen king queen royal throne kingdom ruler power
queen king queen king queen king royal throne kingdom ruler power
king queen prince princess royal family kingdom
queen king prince princess royal family kingdom
man woman human male female person
prince princess king queen royal child family
"""

# -------------------------------
# 🔹 PIPELINE
# -------------------------------
words = preprocess(text)

word2idx, idx2word, counts = build_vocab(words)

pairs = generate_pairs(words, word2idx, window=3)

sampler = negative_sampler(counts, word2idx)

model = Word2Vec(len(word2idx), 50)

train(model, pairs, sampler)

normalize(model)

# -------------------------------
# 🔍 RESULTS
# -------------------------------
print("\n🔍 king →", similar(model, "king", word2idx, idx2word))
print("\n🔍 queen →", similar(model, "queen", word2idx, idx2word))

Epoch 20, Loss: 764.4029
Epoch 40, Loss: 719.7735
Epoch 60, Loss: 697.1783
Epoch 80, Loss: 683.4086

🔍 king → [('queen', np.float64(0.7523577624374878)), ('throne', np.float64(0.6782885496546132)), ('power', np.float64(0.658151860076042)), ('ruler', np.float64(0.6395628519507007)), ('royal', np.float64(0.5957645907882629))]

🔍 queen → [('king', np.float64(0.7523577624374878)), ('child', np.float64(0.6420691760165621)), ('throne', np.float64(0.639528611489996)), ('power', np.float64(0.6214490495567305)), ('ruler', np.float64(0.6202026909919881))]


In [10]:
# =========================================================
# 🚀 PREPARE TEXT FROM YOUR DATASET (WORKING 100%)
# =========================================================

import pandas as pd
import re

# load dataset
df = pd.read_csv("infosys_googlenews.csv")

# -------------------------------
# 🔹 CLEAN TEXT FUNCTION
# -------------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove numbers/symbols
    return text

# use title (since full_text missing)
df['clean_text'] = df['title'].apply(clean_text)

# -------------------------------
# 🔹 LIMIT TO 100 NEWS
# -------------------------------
df = df.head(100)

# -------------------------------
# 🔹 CREATE CORPUS
# -------------------------------
corpus = " ".join(df['clean_text'].tolist())
words = corpus.split()

print("Total words:", len(words))
print("Sample:", words[:20])

Total words: 1412
Sample: ['infosys', 'is', 'the', 'fastest', 'growing', 'it', 'services', 'brand', 'globally', 'with', 'a', 'cagr', 'of', 'in', 'brand', 'value', 'thewirein', 'infosys', 'lifts', 'fy']


In [11]:
# =========================================================
# 🚀 FULL PIPELINE: DATA → CLEAN → WORD2VEC (FINAL)
# =========================================================

import pandas as pd
import numpy as np
import re
from collections import Counter

np.random.seed(42)

# -------------------------------
# 🔹 1. LOAD DATA (LIMIT 100)
# -------------------------------
df = pd.read_csv("infosys_googlenews.csv").head(100)

# -------------------------------
# 🔹 2. CLEAN TEXT
# -------------------------------
def clean_text(text):
    stopwords = set([
        'is','the','a','an','and','of','to','in','on','for','with','at','by','from'
    ])

    words = str(text).lower().split()
    clean = []

    for w in words:
        w = re.sub(r'[^a-z]', '', w)  # remove symbols

        if len(w) < 3:
            continue
        if w in stopwords:
            continue

        clean.append(w)

    return clean

# apply cleaning
all_words = []
for t in df['title']:
    all_words.extend(clean_text(t))

# OPTIONAL BOOST (helps small data)
all_words = all_words * 10

print("Total cleaned words:", len(all_words))


# -------------------------------
# 🔹 3. BUILD VOCAB
# -------------------------------
word_counts = Counter(all_words)
vocab = list(word_counts.keys())

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}


# -------------------------------
# 🔹 4. GENERATE TRAINING PAIRS
# -------------------------------
def generate_pairs(words, window=3):
    pairs = []

    for i, w in enumerate(words):
        center = word2idx[w]

        for j in range(max(0,i-window), min(len(words), i+window+1)):
            if i != j:
                pairs.append((center, word2idx[words[j]]))

    return pairs

pairs = generate_pairs(all_words)


# -------------------------------
# 🔹 5. NEGATIVE SAMPLING
# -------------------------------
def negative_sampler(k=5):
    freqs = np.array([word_counts[w] for w in vocab])
    probs = freqs**0.75
    probs /= probs.sum()

    return np.random.choice(len(vocab), size=k, p=probs)


# -------------------------------
# 🔹 6. MODEL
# -------------------------------
class Word2Vec:
    def __init__(self, V, D):
        self.W = np.random.randn(V, D) / np.sqrt(D)
        self.Wc = np.random.randn(V, D) / np.sqrt(D)


def sigmoid(x):
    return 1/(1+np.exp(-np.clip(x, -10, 10)))


# -------------------------------
# 🔹 7. TRAINING (FIXED)
# -------------------------------
def train(model, pairs, epochs=50, lr=0.005):

    for e in range(epochs):
        loss = 0

        for center, context in pairs:

            vc = model.W[center].copy()
            grad_v = np.zeros_like(vc)

            # POSITIVE
            uo = model.Wc[context]
            score = sigmoid(uo @ vc)
            loss += -np.log(score + 1e-9)

            grad = (score - 1)
            grad_v += grad * uo
            model.Wc[context] -= lr * grad * vc

            # NEGATIVE
            for neg in negative_sampler():
                uk = model.Wc[neg]
                s = sigmoid(uk @ vc)

                loss += -np.log(1 - s + 1e-9)

                grad_n = s
                grad_v += grad_n * uk
                model.Wc[neg] -= lr * grad_n * vc

            # SINGLE UPDATE
            model.W[center] -= lr * grad_v

        if (e+1) % 10 == 0:
            print(f"Epoch {e+1}, Loss: {loss:.4f}")


# -------------------------------
# 🔹 8. NORMALIZE
# -------------------------------
def normalize(model):
    model.W /= (np.linalg.norm(model.W, axis=1, keepdims=True)+1e-9)


# -------------------------------
# 🔹 9. SIMILAR WORDS
# -------------------------------
def similar(word, top_k=5):
    if word not in word2idx:
        return "Word not found"

    vec = model.W[word2idx[word]]

    sims = []
    for i in range(len(model.W)):
        sims.append((idx2word[i], vec @ model.W[i]))

    sims.sort(key=lambda x:-x[1])
    return sims[1:top_k+1]


# =========================================================
# 🚀 RUN MODEL
# =========================================================

model = Word2Vec(len(vocab), 50)

train(model, pairs, epochs=50, lr=0.005)

normalize(model)

# -------------------------------
# 🔍 TEST OUTPUT
# -------------------------------
print("\n🔍 infosys →", similar("infosys"))
print("\n🔍 market →", similar("market"))
print("\n🔍 growth →", similar("growth"))

Total cleaned words: 11060
Epoch 10, Loss: 98741.1272
Epoch 20, Loss: 72403.6184
Epoch 30, Loss: 68031.8507
Epoch 40, Loss: 66730.3558
Epoch 50, Loss: 65788.6249

🔍 infosys → [('shares', np.float64(0.3812613883734437)), ('building', np.float64(0.36953697956058684)), ('early', np.float64(0.3537516414234352)), ('whalesbook', np.float64(0.35300985971214216)), ('forecast', np.float64(0.34332093720071544))]

🔍 market → [('vie', np.float64(0.6178186771628166)), ('benchmarks', np.float64(0.6122007506594996)), ('era', np.float64(0.5882042941572456)), ('recap', np.float64(0.5652998159192308)), ('agency', np.float64(0.5572484170300068))]

🔍 growth → [('revival', np.float64(0.6300753634599321)), ('currency', np.float64(0.6138898958639533)), ('upbeat', np.float64(0.6089762633441917)), ('constant', np.float64(0.5820070205027137)), ('strengthen', np.float64(0.5734531427881571))]


In [12]:
# =========================================================
# 🚀 FINAL PIPELINE: CLEAN → FILTER → WORD2VEC (RESEARCH)
# =========================================================

import pandas as pd
import numpy as np
import re
import random
from collections import Counter

np.random.seed(42)

# -------------------------------
# 🔹 1. LOAD DATA (100 NEWS)
# -------------------------------
df = pd.read_csv("infosys_googlenews.csv").head(100)

# -------------------------------
# 🔹 2. CLEAN TEXT
# -------------------------------
def clean_text(text):
    stopwords = {
        'is','the','a','an','and','of','to','in','on','for','with','at','by','from'
    }

    words = str(text).lower().split()
    clean = []

    for w in words:
        w = re.sub(r'[^a-z]', '', w)

        if len(w) < 3:
            continue
        if w in stopwords:
            continue

        clean.append(w)

    return clean


# collect words
all_words = []
for t in df['title']:
    all_words.extend(clean_text(t))


# -------------------------------
# 🔹 3. BUILD VOCAB
# -------------------------------
word_counts = Counter(all_words)

# -------------------------------
# 🔹 4. REMOVE RARE WORDS
# -------------------------------
min_count = 3
filtered_words = [w for w in all_words if word_counts[w] >= min_count]

print("After rare-word filtering:", len(filtered_words))


# -------------------------------
# 🔹 5. SUBSAMPLING (CRITICAL)
# -------------------------------
def subsample(words):
    total = len(words)
    freqs = {w: word_counts[w]/total for w in word_counts}

    t = 1e-5
    new_words = []

    for w in words:
        prob = 1 - np.sqrt(t / freqs[w])
        if random.random() > prob:
            new_words.append(w)

    return new_words

filtered_words = subsample(filtered_words)

print("After subsampling:", len(filtered_words))


# -------------------------------
# 🔹 6. BOOST SMALL DATA
# -------------------------------
filtered_words = filtered_words * 5


# -------------------------------
# 🔹 7. BUILD FINAL VOCAB
# -------------------------------
vocab = list(set(filtered_words))

word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}


# -------------------------------
# 🔹 8. GENERATE TRAINING PAIRS
# -------------------------------
def generate_pairs(words, window=4):
    pairs = []

    for i, w in enumerate(words):
        center = word2idx[w]

        for j in range(max(0,i-window), min(len(words), i+window+1)):
            if i != j:
                pairs.append((center, word2idx[words[j]]))

    return pairs

pairs = generate_pairs(filtered_words)


# -------------------------------
# 🔹 9. NEGATIVE SAMPLING
# -------------------------------
freqs = np.array([word_counts[w] for w in vocab])
probs = freqs**0.75
probs /= probs.sum()

def negative_sampler(k=5):
    return np.random.choice(len(vocab), size=k, p=probs)


# -------------------------------
# 🔹 10. MODEL
# -------------------------------
class Word2Vec:
    def __init__(self, V, D):
        self.W = np.random.randn(V, D) / np.sqrt(D)
        self.Wc = np.random.randn(V, D) / np.sqrt(D)


def sigmoid(x):
    return 1/(1+np.exp(-np.clip(x, -10, 10)))


# -------------------------------
# 🔹 11. TRAINING (FIXED)
# -------------------------------
def train(model, pairs, epochs=50, lr=0.005):

    for e in range(epochs):
        loss = 0

        for center, context in pairs:

            vc = model.W[center].copy()
            grad_v = np.zeros_like(vc)

            # POSITIVE
            uo = model.Wc[context]
            score = sigmoid(uo @ vc)
            loss += -np.log(score + 1e-9)

            grad = (score - 1)
            grad_v += grad * uo
            model.Wc[context] -= lr * grad * vc

            # NEGATIVE
            for neg in negative_sampler():
                uk = model.Wc[neg]
                s = sigmoid(uk @ vc)

                loss += -np.log(1 - s + 1e-9)

                grad_n = s
                grad_v += grad_n * uk
                model.Wc[neg] -= lr * grad_n * vc

            # SINGLE UPDATE
            model.W[center] -= lr * grad_v

        if (e+1) % 10 == 0:
            print(f"Epoch {e+1}, Loss: {loss:.2f}")


# -------------------------------
# 🔹 12. NORMALIZE
# -------------------------------
def normalize(model):
    model.W /= (np.linalg.norm(model.W, axis=1, keepdims=True)+1e-9)


# -------------------------------
# 🔹 13. SIMILARITY
# -------------------------------
def similar(word, top_k=5):
    if word not in word2idx:
        return "Word not found"

    vec = model.W[word2idx[word]]

    sims = []
    for i in range(len(model.W)):
        sims.append((idx2word[i], vec @ model.W[i]))

    sims.sort(key=lambda x:-x[1])
    return sims[1:top_k+1]


# =========================================================
# 🚀 RUN MODEL
# =========================================================

model = Word2Vec(len(vocab), 50)

train(model, pairs, epochs=50, lr=0.005)

normalize(model)

# -------------------------------
# 🔍 TEST OUTPUT
# -------------------------------
print("\n🔍 infosys →", similar("infosys"))
print("\n🔍 market →", similar("market"))
print("\n🔍 growth →", similar("growth"))

After rare-word filtering: 700
After subsampling: 21
Epoch 10, Loss: 2078.45
Epoch 20, Loss: 1913.05
Epoch 30, Loss: 1819.17
Epoch 40, Loss: 1775.76
Epoch 50, Loss: 1750.12

🔍 infosys → Word not found

🔍 market → Word not found

🔍 growth → Word not found
